In [1]:
import os
import sys
import time
import re
import requests
import gspread

from io import BytesIO
from datetime import datetime
from PIL import Image, ImageOps, UnidentifiedImageError
from google.oauth2.service_account import Credentials
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ==================
# 初期設定
# ==================

start_time = time.time()
now = datetime.now()

# Jupyter・pyどちらも対応
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

sys.path.append(
    os.path.abspath(
        os.path.join(base_dir, "..")
    )
)

from utils.config import PROJECT_DIR


# ==================
# パス設定
# ==================

if os.name == "nt":
    user_base = os.path.join(
        os.environ["USERPROFILE"],
        "myenv310",
        PROJECT_DIR,
    )
else:
    user_base = os.path.join(
        os.path.expanduser("~"),
        "myenv310",
        PROJECT_DIR,
    )


# GoogleサービスアカウントJSON
json_file_path = os.path.join(
    user_base,
    "credentials",
    "gspread-test-402206-686fe226262e.json",
)


# 画像保存先
output_dir = os.path.join(
    user_base,
    "img",
    "machines",
)

os.makedirs(
    output_dir,
    exist_ok=True,
)


# ==================
# スプレッドシート設定
# ==================

GSHEET_NAME = "実機相場サイト"
SHEET_NAME = "slot"


# ==================
# ダウンロード設定
# ==================

# False:
# すでに同じ番号の画像がある場合はスキップ
OVERWRITE_EXISTING = False

# リクエスト間隔
REQUEST_INTERVAL = 0.15

# 通信タイムアウト
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 30

# WebP保存品質
WEBP_QUALITY = 90

# 最大画像サイズ
MAX_IMAGE_SIZE = 20 * 1024 * 1024

# 進捗表示間隔
PROGRESS_INTERVAL = 50


# ==================
# Google Sheets接続
# ==================

def connect_spreadsheet():
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive.readonly",
    ]

    credentials = Credentials.from_service_account_file(
        json_file_path,
        scopes=scopes,
    )

    client = gspread.authorize(
        credentials
    )

    spreadsheet = client.open(
        GSHEET_NAME
    )

    worksheet = spreadsheet.worksheet(
        SHEET_NAME
    )

    return worksheet


# ==================
# HTTPセッション
# ==================

def create_session():
    session = requests.Session()

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=[
            "GET",
        ],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry,
        pool_connections=10,
        pool_maxsize=10,
    )

    session.mount(
        "http://",
        adapter,
    )

    session.mount(
        "https://",
        adapter,
    )

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/150.0.0.0 "
            "Safari/537.36"
        ),
        "Accept": (
            "image/avif,"
            "image/webp,"
            "image/apng,"
            "image/*,"
            "*/*;q=0.8"
        ),
        "Referer": "https://www.p-world.co.jp/",
    })

    return session


# ==================
# 値整形
# ==================

def normalize_machine_id(value):
    """
    A列の値を画像ファイル名用に整える。

    例:
    3209.0 → 3209
    """

    if value is None:
        return ""

    value = str(value).strip()

    if not value:
        return ""

    # 3209.0 → 3209
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]

    # Windowsでファイル名に使えない文字を置換
    value = re.sub(
        r'[\\/:*?"<>|]',
        "_",
        value,
    )

    return value


def normalize_url(value):
    """
    C列のURLを整える。
    """

    if value is None:
        return ""

    url = str(value).strip()

    if not url:
        return ""

    if not url.startswith(
        (
            "http://",
            "https://",
        )
    ):
        return ""

    return url


# ==================
# WebP保存
# ==================

def save_webp(
    image_data,
    output_path,
):
    """
    取得画像をWebPへ変換して保存する。
    """

    temporary_path = (
        output_path + ".tmp"
    )

    try:
        with Image.open(
            BytesIO(image_data)
        ) as image:

            # EXIFの回転情報を反映
            image = ImageOps.exif_transpose(
                image
            )

            # アニメーション画像なら先頭フレーム
            if getattr(
                image,
                "is_animated",
                False,
            ):
                image.seek(0)

            # WebPで保存可能なモードへ変換
            if image.mode in (
                "RGBA",
                "LA",
            ):
                image = image.convert(
                    "RGBA"
                )
            else:
                image = image.convert(
                    "RGB"
                )

            image.save(
                temporary_path,
                format="WEBP",
                quality=WEBP_QUALITY,
                method=6,
            )

        # 保存成功後に本来のファイル名へ置換
        os.replace(
            temporary_path,
            output_path,
        )

        return True, ""

    except UnidentifiedImageError:
        return (
            False,
            "画像データではない",
        )

    except Exception as error:
        return (
            False,
            f"画像保存エラー: {error}",
        )

    finally:
        if os.path.exists(
            temporary_path
        ):
            try:
                os.remove(
                    temporary_path
                )
            except OSError:
                pass


# ==================
# 画像ダウンロード
# ==================

def download_image(
    session,
    image_url,
    output_path,
):
    """
    プロキシを使わず直接接続で画像を取得する。
    """

    try:
        response = session.get(
            image_url,
            timeout=(
                CONNECT_TIMEOUT,
                READ_TIMEOUT,
            ),
            allow_redirects=True,
        )

    except requests.Timeout:
        return (
            False,
            "通信タイムアウト",
        )

    except requests.ConnectionError as error:
        return (
            False,
            f"接続エラー: {error}",
        )

    except requests.RequestException as error:
        return (
            False,
            f"通信エラー: {error}",
        )

    # 画像なし
    if response.status_code in (
        404,
        410,
    ):
        return (
            False,
            f"画像なし HTTP {response.status_code}",
        )

    if response.status_code != 200:
        return (
            False,
            f"HTTP {response.status_code}",
        )

    content_type = response.headers.get(
        "Content-Type",
        "",
    ).lower()

    # HTMLなどが返された場合
    if not content_type.startswith(
        "image/"
    ):
        return (
            False,
            f"画像ではない応答: {content_type}",
        )

    if not response.content:
        return (
            False,
            "画像データが空",
        )

    if len(response.content) > MAX_IMAGE_SIZE:
        return (
            False,
            "画像サイズ上限超過",
        )

    return save_webp(
        response.content,
        output_path,
    )


# ==================
# メイン処理
# ==================

def main():
    print("=" * 80)
    print("P-WORLD画像ダウンロード")
    print(
        f"開始日時: "
        f"{now:%Y-%m-%d %H:%M:%S}"
    )
    print(
        f"シート: "
        f"{GSHEET_NAME} / {SHEET_NAME}"
    )
    print(
        f"保存先: "
        f"{output_dir}"
    )
    print(
        "プロキシ: 使用しない"
    )
    print("=" * 80)

    worksheet = connect_spreadsheet()

    # A列からC列をまとめて取得
    #
    # A列:
    # 保存する画像ファイル名
    #
    # C列:
    # ダウンロードする画像URL
    rows = worksheet.get(
        "A4:C",
        value_render_option=(
            "UNFORMATTED_VALUE"
        ),
    )

    print(
        f"取得行数: "
        f"{len(rows):,}件"
    )

    session = create_session()

    saved_count = 0
    existing_count = 0
    no_id_count = 0
    no_url_count = 0
    no_image_count = 0
    error_count = 0
    processed_count = 0

    failed_rows = []

    try:
        for sheet_row, row in enumerate(
            rows,
            start=4,
        ):
            machine_id = normalize_machine_id(
                row[0]
                if len(row) >= 1
                else ""
            )

            image_url = normalize_url(
                row[2]
                if len(row) >= 3
                else ""
            )

            # A列が空
            if not machine_id:
                no_id_count += 1
                continue

            # C列が空
            if not image_url:
                no_url_count += 1
                continue

            processed_count += 1

            output_path = os.path.join(
                output_dir,
                f"{machine_id}.webp",
            )

            # 既存ファイルはスキップ
            if (
                not OVERWRITE_EXISTING
                and os.path.isfile(output_path)
                and os.path.getsize(output_path) > 0
            ):
                existing_count += 1

                if (
                    processed_count
                    % PROGRESS_INTERVAL
                    == 0
                ):
                    print(
                        f"[進捗] "
                        f"処理={processed_count:,} / "
                        f"保存={saved_count:,} / "
                        f"既存={existing_count:,} / "
                        f"画像なし={no_image_count:,} / "
                        f"エラー={error_count:,}"
                    )

                continue

            success, message = download_image(
                session=session,
                image_url=image_url,
                output_path=output_path,
            )

            if success:
                saved_count += 1

                print(
                    f"[保存] "
                    f"行{sheet_row}: "
                    f"{machine_id}.webp"
                )

            else:
                if "画像なし" in message:
                    no_image_count += 1

                    print(
                        f"[なし] "
                        f"行{sheet_row}: "
                        f"ID={machine_id} / "
                        f"{message}"
                    )

                else:
                    error_count += 1

                    print(
                        f"[エラー] "
                        f"行{sheet_row}: "
                        f"ID={machine_id} / "
                        f"{message}"
                    )

                failed_rows.append([
                    sheet_row,
                    machine_id,
                    image_url,
                    message,
                ])

            if (
                processed_count
                % PROGRESS_INTERVAL
                == 0
            ):
                print(
                    f"[進捗] "
                    f"処理={processed_count:,} / "
                    f"保存={saved_count:,} / "
                    f"既存={existing_count:,} / "
                    f"画像なし={no_image_count:,} / "
                    f"エラー={error_count:,}"
                )

            time.sleep(
                REQUEST_INTERVAL
            )

    except KeyboardInterrupt:
        print(
            "\n[中断] "
            "ユーザー操作により処理を中断しました。"
        )

    finally:
        session.close()

    # ==================
    # 失敗ログ
    # ==================

    failed_log_path = os.path.join(
        output_dir,
        "image_download_failed.tsv",
    )

    if failed_rows:
        with open(
            failed_log_path,
            "w",
            encoding="utf-8-sig",
            newline="",
        ) as file:
            file.write(
                "row\t"
                "machine_id\t"
                "url\t"
                "reason\n"
            )

            for item in failed_rows:
                cleaned_values = []

                for value in item:
                    cleaned_value = (
                        str(value)
                        .replace("\t", " ")
                        .replace("\r", " ")
                        .replace("\n", " ")
                    )

                    cleaned_values.append(
                        cleaned_value
                    )

                file.write(
                    "\t".join(
                        cleaned_values
                    )
                    + "\n"
                )

        print(
            f"失敗ログ: "
            f"{failed_log_path}"
        )

    else:
        # 前回の失敗ログが残っていたら削除
        if os.path.exists(
            failed_log_path
        ):
            try:
                os.remove(
                    failed_log_path
                )
            except OSError:
                pass

    elapsed_time = (
        time.time()
        - start_time
    )

    print("=" * 80)
    print("処理結果")
    print("-" * 80)
    print(
        f"シート取得行数: "
        f"{len(rows):,}件"
    )
    print(
        f"処理対象      : "
        f"{processed_count:,}件"
    )
    print(
        f"新規保存      : "
        f"{saved_count:,}件"
    )
    print(
        f"既存スキップ  : "
        f"{existing_count:,}件"
    )
    print(
        f"A列番号なし   : "
        f"{no_id_count:,}件"
    )
    print(
        f"C列URLなし    : "
        f"{no_url_count:,}件"
    )
    print(
        f"URL先画像なし : "
        f"{no_image_count:,}件"
    )
    print(
        f"その他エラー  : "
        f"{error_count:,}件"
    )
    print(
        f"処理時間      : "
        f"{elapsed_time:.1f}秒"
    )
    print(
        f"保存先        : "
        f"{output_dir}"
    )
    print("=" * 80)


if __name__ == "__main__":
    main()

P-WORLD画像ダウンロード
開始日時: 2026-07-28 15:37:42
シート: 実機相場サイト / slot
保存先: C:\Users\stray\myenv310\soubanavi\img\machines
プロキシ: 使用しない
取得行数: 3,209件
[保存] 行4: 3209.webp
[保存] 行5: 3208.webp
[保存] 行6: 3207.webp
[保存] 行7: 3206.webp
[保存] 行8: 3205.webp
[保存] 行9: 3204.webp
[保存] 行10: 3203.webp
[保存] 行11: 3202.webp
[保存] 行12: 3201.webp
[保存] 行13: 3200.webp
[保存] 行14: 3199.webp
[保存] 行15: 3198.webp
[保存] 行16: 3197.webp
[保存] 行17: 3196.webp
[保存] 行18: 3195.webp
[保存] 行19: 3194.webp
[保存] 行20: 3193.webp
[保存] 行21: 3192.webp
[保存] 行22: 3191.webp
[保存] 行23: 3190.webp
[保存] 行24: 3189.webp
[保存] 行25: 3188.webp
[保存] 行26: 3187.webp
[保存] 行27: 3186.webp
[保存] 行28: 3185.webp
[保存] 行29: 3184.webp
[保存] 行30: 3183.webp
[保存] 行31: 3182.webp
[保存] 行32: 3181.webp
[保存] 行33: 3180.webp
[保存] 行34: 3179.webp
[保存] 行35: 3178.webp
[保存] 行36: 3177.webp
[保存] 行37: 3176.webp
[保存] 行38: 3175.webp
[保存] 行39: 3174.webp
[保存] 行40: 3173.webp
[保存] 行41: 3172.webp
[保存] 行42: 3171.webp
[保存] 行43: 3170.webp
[保存] 行44: 3169.webp
[保存] 行45: 3168.webp
[保存] 行46: 3167.webp
[保存] 行4